# Multi-Model YOLO Validation and Reporting

Run the `run_yolo_validation_report.py` script on multiple YOLO models, collect metrics, and compare results.


In [ ]:
# 1. Set Up Environment and Install Dependencies

import os

import sys

from pathlib import Path



import torch

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



print(f"Python: {sys.version}")

print(f"Torch version: {torch.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")



# Ensure project root is on sys.path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:

    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# 2. Load and Inspect Existing Script

from pprint import pprint



SCRIPT_PATH = PROJECT_ROOT / "yolo_test" / "run_yolo_validation_report.py"

print(f"Using script: {SCRIPT_PATH}")

if not SCRIPT_PATH.exists():

    raise FileNotFoundError(f"Script not found: {SCRIPT_PATH}")



with open(SCRIPT_PATH, "r") as f:

    script_source = f.read()



print(script_source[:2000])  # show first 2000 characters


In [ ]:
# 3. Define a helper to run the script programmatically

import subprocess



def run_yolo_report_for_model(model_name: str,

                              dataset_name: str = "bdd100k_yolo_limited",

                              split: str = "test",

                              iou: float = 0.5,

                              base_dir: str | None = None) -> subprocess.CompletedProcess:

    """Run the existing script for a single model via subprocess.



    Returns the CompletedProcess so the caller can inspect returncode/stdout/stderr.

    """

    if base_dir is None:

        base_dir = str(PROJECT_ROOT)



    python_executable = str((PROJECT_ROOT / "yolo_project" / "bin" / "python"))

    if not (PROJECT_ROOT / "yolo_project" / "bin" / "python").exists():

        # Fallback to current Python if venv python is not found

        python_executable = sys.executable



    cmd = [

        python_executable,

        str(SCRIPT_PATH),

        "--model-name", model_name,

        "--dataset-name", dataset_name,

        "--split", split,

        "--iou", str(iou),

        "--base-dir", base_dir,

    ]



    print("Running:", " ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("Return code:", result.returncode)

    if result.stdout:

        print("--- STDOUT (tail) ---")

        print("\n".join(result.stdout.splitlines()[-20:]))

    if result.stderr:

        print("--- STDERR (tail) ---")

        print("\n".join(result.stderr.splitlines()[-20:]))

    return result


In [ ]:
# 4. Define model configuration list

MODEL_CONFIGS = [

    {"name": "yolov8n", "dataset": "bdd100k_yolo_limited", "split": "test", "iou": 0.5},

    {"name": "yolov8s", "dataset": "bdd100k_yolo_limited", "split": "test", "iou": 0.5},

    {"name": "yolov8m", "dataset": "bdd100k_yolo_limited", "split": "test", "iou": 0.5},

]



MODEL_CONFIGS


In [ ]:
# 5–6. Loop over models, run script, and collect metrics

import json



results_summary: list[dict[str, object]] = []



for cfg in MODEL_CONFIGS:

    print("=" * 80)

    print(f"Running model: {cfg['name']} | dataset={cfg['dataset']} | split={cfg['split']} | IoU={cfg['iou']}")

    print("=" * 80)

    proc = run_yolo_report_for_model(

        model_name=cfg["name"],

        dataset_name=cfg["dataset"],

        split=cfg["split"],

        iou=cfg["iou"],

        base_dir=str(PROJECT_ROOT),

    )



    if proc.returncode != 0:

        print(f"⚠️ Model {cfg['name']} failed, skipping metrics load.")

        results_summary.append({

            "model_name": cfg["name"],

            "dataset": cfg["dataset"],

            "split": cfg["split"],

            "iou": cfg["iou"],

            "status": "error",

        })

        continue



    # Locate the most recent run directory for this model

    runs_root = PROJECT_ROOT / "yolo_test" / "runs"

    pattern = f"{cfg['name']}_testing_"

    model_runs = [p for p in runs_root.glob(f"{cfg['name']}_testing_*") if p.is_dir()]

    if not model_runs:

        print(f"⚠️ No run directory found for {cfg['name']}")

        continue

    latest_run = sorted(model_runs)[-1]

    metrics_path = latest_run / "metrics_data.json"

    if not metrics_path.exists():

        print(f"⚠️ metrics_data.json not found for {cfg['name']} at {metrics_path}")

        continue



    with open(metrics_path, "r") as f:

        metrics_data = json.load(f)



    overall = metrics_data["custom_metrics"]["overall"]

    yolo_overall = metrics_data["yolo_official_metrics"]["overall"]



    results_summary.append({

        "model_name": cfg["name"],

        "dataset": cfg["dataset"],

        "split": cfg["split"],

        "iou": cfg["iou"],

        "precision_confusion": overall["precision"],

        "recall_confusion": overall["recall"],

        "f1_confusion": overall["f1_score"],

        "precision_yolo": yolo_overall["precision"],

        "recall_yolo": yolo_overall["recall"],

        "map50": yolo_overall["map50"],

        "map50_95": yolo_overall["map50_95"],

        "status": "ok",

        "run_dir": str(latest_run),

    })



results_df = pd.DataFrame(results_summary)

results_df

In [ ]:
# 7. Compare results across models (tables and plots)

if not results_df.empty:

    display(results_df)



    sns.set_style("whitegrid")

    plt.figure(figsize=(10, 6))

    sns.barplot(data=results_df, x="model_name", y="map50")

    plt.title("mAP@0.5 per model (YOLO official)")

    plt.ylabel("mAP@0.5")

    plt.show()



    plt.figure(figsize=(10, 6))

    sns.barplot(data=results_df, x="model_name", y="f1_confusion")

    plt.title("F1 (from confusion matrix) per model")

    plt.ylabel("F1 score")

    plt.show()

else:

    print("No successful runs to compare.")


In [ ]:
# 8. Demonstrate CLI usage from notebook (optional)

# Example: run a single model via CLI-style command

import shlex



single_cmd = f"{sys.executable} {SCRIPT_PATH} --model-name yolov8n --dataset-name bdd100k_yolo_limited --split test --iou 0.5 --base-dir {PROJECT_ROOT}"

print("CLI example:")

print(single_cmd)

# You can uncomment the next line to actually run it from the notebook:

# !python {SCRIPT_PATH} --model-name yolov8n --dataset-name bdd100k_yolo_limited --split test --iou 0.5 --base-dir {PROJECT_ROOT}


In [ ]:
# 9. (Optional) Placeholder for tests integration

print("For VS Code Test Explorer integration, you can create pytest tests \n"

      "that import run_yolo_validation_report.main or call the script \n"

      "on a very small dummy dataset, then run !pytest from here.")
